In [1]:
!pip -q install tensorflow scikit-learn matplotlib seaborn

In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


**Hyperparameters**

In [3]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

VOCAB_SIZE = 20000
MAX_LEN = 250
EMBED_DIM = 128
LSTM_UNITS = 128
BATCH_SIZE = 32
EPOCHS = 5
VAL_SIZE = 5000

**Load dataset**

In [4]:
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print("Train samples:", len(x_train))
print("Test samples :", len(x_test))
print("Example encoded review:", x_train[0][:20])
print("Example label:", y_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train samples: 25000
Test samples : 25000
Example encoded review: [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
Example label: 1


**Pad sequences**

In [5]:
x_train = keras.preprocessing.sequence.pad_sequences(
    x_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

x_test = keras.preprocessing.sequence.pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("x_train shape:", x_train.shape)
print("x_test shape :", x_test.shape)

x_train shape: (25000, 250)
x_test shape : (25000, 250)


**Create validation set**

In [6]:
x_val = x_train[:VAL_SIZE]
y_val = y_train[:VAL_SIZE]

x_train_partial = x_train[VAL_SIZE:]
y_train_partial = y_train[VAL_SIZE:]

print("Train:", x_train_partial.shape, y_train_partial.shape)
print("Val  :", x_val.shape, y_val.shape)
print("Test :", x_test.shape, y_test.shape)

Train: (20000, 250) (20000,)
Val  : (5000, 250) (5000,)
Test : (25000, 250) (25000,)


**Build LSTM model**

In [7]:
lstm_model = keras.Sequential([
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    layers.SpatialDropout1D(0.2),
    layers.LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Train model**

In [8]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start_time = time.time()

history = lstm_model.fit(
    x_train_partial,
    y_train_partial,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

train_time = time.time() - start_time
print(f"Training time: {train_time:.2f} seconds")

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 363s 565ms/step - accuracy: 0.5034 - loss: 0.6945 - val_accuracy: 0.4926 - val_loss: 0.7016
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 338s 541ms/step - accuracy: 0.5359 - loss: 0.6856 - val_accuracy: 0.4948 - val_loss: 0.7172
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 399s 569ms/step - accuracy: 0.5870 - loss: 0.6319 - val_accuracy: 0.5606 - val_loss: 0.6870
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 345s 553ms/step - accuracy: 0.7071 - loss: 0.5101 - val_accuracy: 0.8198 - val_loss: 0.4434
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 356s 570ms/step - accuracy: 0.8995 - loss: 0.2545 - val_accuracy: 0.8608 - val_loss: 0.3629
Training time: 1801.86 seconds


**Evaluate on test set**

In [9]:
test_loss, test_acc = lstm_model.evaluate(
    x_test,
    y_test,
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Test Loss    :", round(test_loss, 4))
print("Test Accuracy:", round(test_acc, 4))

Test Loss    : 0.3898
Test Accuracy: 0.851


**Predict and compute metrics**

In [10]:
y_pred_prob = lstm_model.predict(x_test, batch_size=BATCH_SIZE, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(test_acc, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

Accuracy : 0.851
Precision: 0.8664
Recall   : 0.83
F1-score : 0.8478
